In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Load data from CSV
csv_path = 'full_data.csv'
df = pd.read_csv(csv_path, low_memory=False)

# Use full_aa_sequence as x and score as y
data = np.array([[row['full_aa_sequence'], row['score']] for _, row in df.iterrows()], dtype=object)

# Make dict of data with sequence as key and score as value
data_dict = {}
for i in range(len(data)):
    key = data[i][0]
    value = data[i][1]
    data_dict[key] = value
np.save('data_dict.npy', data_dict)
np.save('data.npy', data)

print(f"Loaded {len(data)} sequences")
print(f"Example sequence (first 100 chars): {data[0][0][:100]}")
print(f"Example score: {data[0][1]}")

In [ ]:
# Check unique characters in sequences
all_chars = set()
for seq in df['full_aa_sequence']:
    all_chars.update(set(seq))
print(f"Unique characters in sequences: {sorted(all_chars)}")
print(f"Number of unique characters: {len(all_chars)}")

In [ ]:
# Finding the largest protein sequence
def find_max_len(data):
    max_len = 0
    for i in range(data.shape[0]):
        str_len = len(data[i, 0])
        if max_len < str_len:
            max_len = str_len
    print('Max len is', max_len)
    return max_len

max_len = find_max_len(data)

In [ ]:
# Standard 20 amino acids
amino_acid = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
print('Number of unique amino acids are', len(amino_acid))
np.save('categorical_variables', amino_acid)

def onehotseq(sequence):
    aa_seq = ''
    seq_len = len(sequence)
    seq_en = np.zeros((seq_len, len(amino_acid)))
    act_len = 0
    for i in range(seq_len):
        if sequence[i] in amino_acid:
            pos = amino_acid.index(sequence[i])
            seq_en[act_len, pos] = 1
            act_len += 1  
            aa_seq += sequence[i]    
    return seq_en[0:act_len, :], aa_seq

In [ ]:
# One-hot encode all sequences
ohe = np.zeros((data.shape[0], max_len, len(amino_acid)))  # batch_size x sequence_len x 20
seq_string = np.zeros((data.shape[0],), dtype=object)  # store string sequences
seq_lengths = np.zeros((data.shape[0],))
scores = np.zeros((data.shape[0], 1))

for i in range(ohe.shape[0]):
    if i % 50000 == 0:
        print(f"Processing {i}/{ohe.shape[0]}")
    seq_en, aa_seq = onehotseq(data[i, 0])
    seq_string[i] = aa_seq
    ohe[i, 0:seq_en.shape[0], :] = seq_en
    seq_lengths[i] = seq_en.shape[0]
    scores[i, 0] = data[i, 1]

# Normalize scores to [0, 1]
score_min = scores.min()
score_max = scores.max()
output_y = (scores - score_min) / (score_max - score_min)
seq_length = seq_lengths

print(f"\nTotal samples before filtering: {len(ohe)}")
print(f"Score range: [{score_min}, {score_max}] -> normalized to [0, 1]")

# Filter to keep only sequences with y <= 0.2
cutoff = 0.2
filter_idx = output_y.flatten() <= cutoff
ohe = ohe[filter_idx]
seq_string = seq_string[filter_idx]
seq_length = seq_length[filter_idx]
output_y = output_y[filter_idx]

print(f"\nAfter filtering (y <= {cutoff}): {len(ohe)} samples")
print(f"Y range after filtering: min={output_y.min():.4f}, max={output_y.max():.4f}")

In [ ]:
# Split into train/test/valid (70/15/15)
all_ex = np.arange(ohe.shape[0])
x_train, x_temp, _, _ = train_test_split(all_ex, all_ex, test_size=0.3, random_state=50)
x_valid, x_test, _, _ = train_test_split(x_temp, x_temp, test_size=0.5, random_state=50)

print('Train', x_train.shape)
print('Test', x_test.shape)
print('Valid', x_valid.shape)
print(f'OHE shape: {ohe[x_train].shape}, Length shape: {seq_length[x_train].shape}, Y shape: {output_y[x_train].shape}')

In [ ]:
np.save('./x_train', ohe[x_train])
np.save('./len_train', seq_length[x_train])
np.save('./y_train', output_y[x_train])
np.save('./seq_train', seq_string[x_train])

np.save('./x_valid', ohe[x_valid])
np.save('./len_valid', seq_length[x_valid])
np.save('./y_valid', output_y[x_valid])
np.save('./seq_valid', seq_string[x_valid])

np.save('./x_test', ohe[x_test])
np.save('./len_test', seq_length[x_test])
np.save('./y_test', output_y[x_test])
np.save('./seq_test', seq_string[x_test])

print("Data files saved successfully!")
print(f"x_train shape: {ohe[x_train].shape}")
print(f"x_valid shape: {ohe[x_valid].shape}")
print(f"x_test shape: {ohe[x_test].shape}")

In [ ]:
plt.hist(seq_length)
plt.title('Sequence Length Distribution')
plt.xlabel('Length')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.hist(output_y, bins=50)
plt.title('Normalized Score Distribution')
plt.xlabel('Score (normalized)')
plt.ylabel('Count')
plt.show()